In [2]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [9]:
pathData = '/glade/u/home/demurray/External File Uploads/DOC data'
os.chdir(pathData)

#First read in NADP archived DOC data and site metadata and merge these together
ntn = pd.read_csv('ntn.csv')
ntn = ntn[['siteId', 'latitude', 'longitude']]
nadp_arch = pd.read_csv('NADP DOM data clean.csv')
nadp_arch = pd.merge(nadp_arch, ntn, on = 'siteId')
nadp_arch = nadp_arch.rename(columns = {'subppt' : 'Precip_mm', 'DOC_mgL_Final_UNH' : 'DOC_mgL'})
nadp_arch

,siteId,dateOn,dateOff,Precip_mm,DOC_mgL,latitude,longitude
0,AK02,2/21/2017 18:05,2/28/2017 18:28,8.128,0.39,58.5139,-134.7843
1,AK02,3/21/2017 18:55,3/28/2017 17:50,12.192,0.49,58.5139,-134.7843
2,AK02,4/18/2017 18:07,4/25/2017 18:01,8.128,0.59,58.5139,-134.7843
3,AK02,5/16/2017 16:18,5/23/2017 18:23,39.116,0.48,58.5139,-134.7843
4,AK02,6/13/2017 18:02,6/20/2017 17:17,78.994,0.43,58.5139,-134.7843
...,...,...,...,...,...,...,...
269,WY08,6/12/2018 21:15,6/18/2018 20:00,41.402,0.59,44.9166,-110.4203
270,WY08,8/21/2018 14:45,8/28/2018 20:15,15.240,1.43,44.9166,-110.4203
271,WY08,10/2/2018 18:45,10/9/2018 15:30,13.462,0.75,44.9166,-110.4203
272,WY08,11/21/2018 16:45,11/26/2018 20:15,9.144,0.41,44.9166,-110.4203


In [10]:
#Next read in the other DOC lat lon file and then data from the LTERs
other_lat_lon = pd.read_csv('DOC data lat lon.csv')
lter_dat = pd.read_csv('DOC_dep_HJ_Niwot_MN16_SR.csv')
lter_dat = pd.merge(lter_dat, other_lat_lon, on = 'siteId')
lter_dat =  lter_dat[['siteId', 'dateOn', 'dateOff', 'Precip_mm', 'DOC-C', 'latitude', 'longitude']]
lter_dat = lter_dat.rename(columns = {'DOC-C' : 'DOC_mgL'})
lter_dat

,siteId,dateOn,dateOff,Precip_mm,DOC_mgL,latitude,longitude
0,OR10,NaN,10/5/2004,0.000,NaN,44.266640,-122.177000
1,OR10,10/5/2004,10/20/2004,30.734,0.55,44.266640,-122.177000
2,OR10,10/20/2004,10/27/2004,62.992,0.42,44.266640,-122.177000
3,OR10,10/27/2004,11/1/2004,31.242,0.72,44.266640,-122.177000
4,OR10,11/1/2004,11/9/2004,29.464,1.01,44.266640,-122.177000
...,...,...,...,...,...,...,...
1780,SR,9/5/2012,9/10/2012,48.000,NaN,44.483333,-72.166667
1781,SR,9/10/2012,9/19/2012,20.600,NaN,44.483333,-72.166667
1782,SR,9/19/2012,9/25/2012,45.500,NaN,44.483333,-72.166667
1783,SR,9/25/2012,10/2/2012,11.700,NaN,44.483333,-72.166667


In [12]:
#Read in Iavoriska 2016 deposition data (this is event-scale so need to groupby day and sum)
ssczo = pd.read_csv('DOCdepdata_Iavoriska2016.csv', parse_dates = ['dateOn'])
ssczo = ssczo[['dateOn', 'timeOn', 'Precip depth (mm)', 'DOC mgC/L']]
ssczo_daily = ssczo.groupby('dateOn')[['Precip depth (mm)', 'DOC mgC/L']].sum().reset_index()
ssczo_daily['dateOff'] = ssczo['dateOn']
ssczo_daily['latitude'] = 40.66598
ssczo_daily['longitude'] = -77.90376
ssczo_daily['siteId'] = 'PA15'
ssczo_daily = ssczo_daily.rename(columns = {'Precip depth (mm)' : 'Precip_mm', 'DOC mgC/L' : 'DOC_mgL'})
ssczo_daily

,dateOn,Precip_mm,DOC_mgL,dateOff,latitude,longitude,siteId
0,2012-05-14,34.7,2.81,2012-05-14,40.66598,-77.90376,PA15
1,2012-05-15,10.7,0.32,2012-05-14,40.66598,-77.90376,PA15
2,2012-05-21,17.3,8.92,2012-05-14,40.66598,-77.90376,PA15
3,2012-05-22,3.2,0.99,2012-05-15,40.66598,-77.90376,PA15
4,2012-05-23,1.2,2.39,2012-05-21,40.66598,-77.90376,PA15
...,...,...,...,...,...,...,...
99,2013-08-08,0.1,4.42,2013-03-31,40.66598,-77.90376,PA15
100,2013-08-09,4.0,3.15,2013-04-10,40.66598,-77.90376,PA15
101,2013-08-13,1.8,6.48,2013-04-10,40.66598,-77.90376,PA15
102,2013-08-19,2.8,1.60,2013-04-10,40.66598,-77.90376,PA15


In [14]:
#Read in TF data
tf = pd.read_csv('Wet Deposition LRHO TF DOC.csv')
tf['latitude'] = 43.108
tf['longitude'] = -70.95
tf = tf.rename(columns = {'DOCmgCL' : 'DOC_mgL'})
tf

,siteId,dateOn,dateOff,Precip_mm,DOC_mgL,latitude,longitude
0,TF,11/18/2003,11/25/2003,8.5,0.993,43.108,-70.95
1,TF,11/25/2003,12/2/2003,13.6,0.739,43.108,-70.95
2,TF,12/2/2003,12/9/2003,24.8,0.385,43.108,-70.95
3,TF,12/9/2003,12/16/2003,77.1,0.308,43.108,-70.95
4,TF,12/16/2003,12/23/2003,11.2,0.357,43.108,-70.95
...,...,...,...,...,...,...,...
1007,TF,12/22/2020,12/29/2020,30.0,0.198,43.108,-70.95
1008,TF,12/29/2020,1/5/2021,16.2,0.416,43.108,-70.95
1009,TF,1/5/2021,1/19/2021,37.1,0.371,43.108,-70.95
1010,TF,1/19/2021,2/1/2021,6.4,0.718,43.108,-70.95


In [16]:
#read in HBEF data
hbef = pd.read_csv('HBEF_DOC data.csv')
hbef['latitude'] = 43.95
hbef['longitude'] = -71.73333333
hbef = hbef[['siteId', 'dateOn', 'dateOff', 'Precip_mm', 'DOC-C', 'latitude', 'longitude']]
hbef = hbef.rename(columns = {'DOC-C' : 'DOC_mgL'})
hbef.dropna(inplace = True)
hbef

,siteId,dateOn,dateOff,Precip_mm,DOC_mgL,latitude,longitude
1,NH02,1/3/2000,1/10/2000,37.4,0.53,43.95,-71.733333
2,NH02,1/10/2000,1/18/2000,32.3,0.42,43.95,-71.733333
3,NH02,1/24/2000,2/1/2000,36.4,0.20,43.95,-71.733333
4,NH02,2/1/2000,2/15/2000,50.0,0.35,43.95,-71.733333
5,NH02,2/15/2000,2/22/2000,24.3,0.24,43.95,-71.733333
...,...,...,...,...,...,...,...
570,NH02,4/28/2014,5/6/2014,39.1,1.40,43.95,-71.733333
571,NH02,5/6/2014,5/12/2014,7.1,2.77,43.95,-71.733333
572,NH02,5/12/2014,5/19/2014,50.5,1.47,43.95,-71.733333
573,NH02,5/19/2014,5/26/2014,27.7,1.72,43.95,-71.733333


In [26]:
#read in BC data
bc_dat = pd.read_csv('BC_data_clean.csv')
bc_dat = pd.merge(bc_dat, ntn, on = 'siteId')
bc_dat = bc_dat[['siteId', 'latitude', 'longitude', 'dateOn', 'dateOff', 'Precip_mm', 'rBC_mgL']]
bc_dat['Variable'] = 'rBC_mgL'
bc_dat = bc_dat.rename(columns = {'rBC_mgL' : 'Value'})
bc_dat

,siteId,latitude,longitude,dateOn,dateOff,Precip_mm,Value,Variable
0,NV03,38.7992,-119.2567,10/27/2020,11/3/2020,3.05,0.0001,rBC_mgL
1,NV03,38.7992,-119.2567,11/7/2020,11/10/2020,24.13,0.0007,rBC_mgL
2,NC06,34.8846,-76.6207,11/17/2020,11/24/2020,4.32,0.0001,rBC_mgL
3,NC06,34.8846,-76.6207,11/3/2020,11/10/2020,6.10,0.0002,rBC_mgL
4,NC06,34.8846,-76.6207,10/28/2020,11/3/2020,12.19,0.0006,rBC_mgL
...,...,...,...,...,...,...,...,...
467,AR16,36.0842,-92.5868,11/17/2020,11/24/2020,14.73,0.0148,rBC_mgL
468,MO05,36.9108,-90.3187,11/10/2020,11/17/2020,18.80,0.0176,rBC_mgL
469,MN27,44.2370,-95.3011,11/3/2020,11/10/2020,29.97,0.0178,rBC_mgL
470,NM07,35.7788,-106.2660,11/17/2020,11/24/2020,13.46,0.0271,rBC_mgL


In [29]:
#concatenate all files together --> should be a total of ~ 4200 data points
doc_df = pd.concat([hbef, tf, ssczo_daily, lter_dat, nadp_arch])
doc_df['Variable'] = 'DOC_mgL'
doc_df = doc_df.rename(columns = {'DOC_mgL' : 'Value'})
doc_bc_df = pd.concat([doc_df, bc_dat])
doc_bc_df

#write master file to read into timeseries pairing
doc_bc_df.to_csv('DOC_BC_wetdep_compiled.csv')